# 02. Препроцессинг: price + deals → joined

Пайплайн:
1. Загрузка `price_filtered.csv` (результат `01_filtering`) + сырые сделки.
2. Переименование колонок RU → EN (`PRICE_RENAME_MAP`, `DEALS_RENAME_MAP`).
3. Составной ключ `unit_match_key` (7 сегментов через `@`).
4. Дедупликация сделок: по `deal_row_key` (точные дубли), затем по `unit_match_key` (первая продажа).
5. **LEFT JOIN** price ← deals → `price_deals_left_unit_match_key.csv`  
   Все строки прайса сохраняются; `registration_date_raw = NaT` для непроданных → таргет = 0.
6. **INNER JOIN** price ∩ deals → `price_deals_inner_unit_match_key.csv`  
   Только лоты, по которым зафиксирована сделка.

**Ключ:** `project_id @ building_id @ section @ conditional_number @ floor @ area @ room_count`

In [1]:
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
DATA_DIR   = REPO_ROOT / "data"
RAW_DIR    = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import FILTERED_PARQUET, JOINED_INNER_CSV
from src.unit_key import (
    UNIT_KEY_PARTS_EN,
    add_unit_match_key,
    fix_price_room_count_from_pool,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Загрузка данных

Прайс: `price_filtered.csv` (уже отфильтрован в `01_filtering`, `month_span == 'begin'`).  
Сделки: сырой CSV из `data/raw/`.  
Параметр `N` — число строк для отладки (`None` = читать целиком).

In [2]:
N = None  # None = полный датасет; int = ограничение строк для отладки

price_raw = pd.read_parquet(FILTERED_PARQUET)
if N is not None:
    price_raw = price_raw.head(N)
deals_raw = pd.read_csv(RAW_DIR / "deals_dataflat_2020_2025.csv", nrows=N, low_memory=False)

print(f"price_filtered: {price_raw.shape}")
print(f"deals_raw:      {deals_raw.shape}")

price_filtered: (4880319, 40)
deals_raw:      (772999, 53)


In [3]:
# Оставляем только проекты, присутствующие И в прайсе, И в сделках
p_pid_col = next((c for c in ("ID Проекта", "project_id") if c in price_raw.columns), None)
d_pid_col = next((c for c in ("ID Проекта", "project_id") if c in deals_raw.columns), None)

if p_pid_col and d_pid_col:
    price_pids  = set(price_raw[p_pid_col].dropna().astype(str).unique())
    deals_pids  = set(deals_raw[d_pid_col].dropna().astype(str).unique())
    common_pids = price_pids & deals_pids

    n_price_before = len(price_raw)
    n_deals_before = len(deals_raw)

    price_raw = price_raw[price_raw[p_pid_col].astype(str).isin(common_pids)].copy()
    deals_raw = deals_raw[deals_raw[d_pid_col].astype(str).isin(common_pids)].copy()

    print(f"Пересечение project_id: {len(common_pids):,} общих проектов "
          f"(price: {len(price_pids):,}, deals: {len(deals_pids):,})")
    print(f"price: {n_price_before:,} → {len(price_raw):,}  "
          f"(−{n_price_before - len(price_raw):,} строк из проектов только в прайсе)")
    print(f"deals: {n_deals_before:,} → {len(deals_raw):,}  "
          f"(−{n_deals_before - len(deals_raw):,} строк из проектов только в сделках)")
else:
    print("ВНИМАНИЕ: не найдена колонка project_id — фильтр пересечения не применён")

Пересечение project_id: 758 общих проектов (price: 1,072, deals: 923)
price: 4,880,319 → 4,659,378  (−220,941 строк из проектов только в прайсе)
deals: 772,999 → 756,791  (−16,208 строк из проектов только в сделках)


## 2. Словари переименования (RU → EN)

Общие для price и deals поля получают одинаковые имена, чтобы `unit_match_key` строился одинаково.

In [4]:
PRICE_RENAME_MAP: dict[str, str] = {
    "pool":                       "pool",
    "ID Проекта":                 "project_id",
    "ID Проекта для окружения":   "env_project_id",
    "ID Корпус":                  "building_id",
    "Проект":                     "project_name",
    "Девелопер":                  "developer",
    "Класс проекта":              "project_class",
    "Регион":                     "region",
    "Округ":                      "macro_district",
    "Район":                      "district",
    "Старт продаж":               "sales_start",
    "Сдача К":                    "completion_k",
    "Ключи":                      "keys_status",
    "Тип помещения":              "premises_type",
    "Секция":                     "section",
    "Этаж":                       "floor",
    "Отделка в отчет":            "finish_in_report",
    "Тип кв/ап":                  "unit_typology",
    "Комнатность":                "room_count",
    "Площадь":                    "area",
    "Цена":                       "price",
    "Цена без скидки":            "price_list",
    "Старый бюджет":              "budget_old",
    "Бюджет без скидки":          "budget_list",
    "Бюджет":                     "budget",
    "Скидка, %":                  "discount_pct",
    "Наличие бюджета":            "has_budget",
    "Изменение цены последнее":   "last_price_change",
    "Дата файла":                 "file_date",
    "Начало/конец месяца":        "month_span",
    "Источник":                   "source",
    "Период":                     "period",
    "lat":                        "lat",
    "lng":                        "lng",
    "Условный номер":             "conditional_number",
    "Отделка К":                  "finish_tier",
    "Отделка текст":              "finish_text",
    "Договор К":                  "contract_type_k",
    "Стадия К":                   "stage_k",
    "Экспозиция":                 "exposure",
}

DEALS_RENAME_MAP: dict[str, str] = {
    "key":                                "deal_row_key",
    "lat":                                "lat",
    "lng":                                "lng",
    "ID Проекта":                         "project_id",
    "ID Проекта для окруженияSales":      "env_project_id_sales",
    "ID Корпус":                          "building_id",
    "Проект":                             "project_name",
    "Девелопер":                          "developer",
    "Класс проекта":                      "project_class",
    "Регион":                             "region",
    "Округ":                              "macro_district",
    "Район":                              "district",
    "Округ Направление":                  "macro_district_direction",
    "Комнатность":                        "room_count",
    "Этаж":                               "floor",
    "Условный номер":                     "conditional_number",
    "Площадь":                            "area",
    "Тип помещения":                      "premises_type",
    "Наличие бюджета":                    "has_budget",
    "Цена":                               "price",
    "Цена без скидки":                    "price_list",
    "Бюджет":                             "budget",
    "Бюджет без скидки":                  "budget_list",
    "Ключи":                              "keys_status",
    "Корпус":                             "building_name",
    "Покупатель ФЛ":                      "buyer_person",
    "Покупатель ЮЛ":                      "buyer_company",
    "Номер регистрации":                  "registration_number",
    "Дата регистрации":                   "registration_date",
    "Дата регистрации_":                  "registration_date_raw",
    "Залогодержатель/Банк":               "pledge_holder_bank",
    "Длительность обременения":           "pledge_duration_months",
    "Тип обременения":                    "pledge_type",
    "Отделка":                            "finish",
    "Дата подписания":                    "signing_date",
    "Дата ДДУ":                           "ddu_date",
    "Срок сдачи":                         "completion_due",
    "Стадия строительства":               "construction_stage",
    "Ипотека":                            "mortgage",
    "Секция":                             "section",
    "Старт продаж":                       "sales_start",
    "Продавец ЮЛ":                        "seller_company",
    "Продавец ФЛ ID":                     "seller_person_id",
    "Описание помещения":                 "premises_description",
    "Опт":                                "is_wholesale",
    "Стадия строительства в дату ДДУ":    "construction_stage_at_ddu",
    "Цена ДДУ":                           "price_ddu",
    "Тип сделки":                         "deal_type",
    "Дата регистрации модель":            "registration_date_model",
    "Цена кв. м":                         "price_per_sqm",
    "ID дом.рф":                          "dom_rf_id",
    "Бюджет по ПД":                       "budget_pd",
    "Цена по ПД":                         "price_pd",
}

## 3. Функции препроцессинга и сборки ключа

`unit_match_key = project_id @ building_id @ section @ conditional_number @ floor @ area @ room_count`

Правила нормализации сегментов — в `src/unit_key.py` (пустые/прочерки → пустой сегмент, `area` round до 0.1, `room_count` целое/`'ст'→1`, `section` strip+запятая→точка). Для price нечисловая `room_count` восстанавливается из последнего сегмента `pool`.

In [5]:
def rename_price(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={c: PRICE_RENAME_MAP[c] for c in df.columns if c in PRICE_RENAME_MAP})


def rename_deals(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={c: DEALS_RENAME_MAP[c] for c in df.columns if c in DEALS_RENAME_MAP})


def preprocess_deals(df: pd.DataFrame) -> pd.DataFrame:
    x = rename_deals(df)
    if "premises_description" in x.columns and "section" in x.columns:
        section_from_desc = x["premises_description"].str.extract(r"секция\s+(\d+)", expand=False)
        mask = x["section"].isna() | (x["section"] == "-")
        x.loc[mask, "section"] = section_from_desc[mask]
    return add_unit_match_key(x)


def preprocess_price(df: pd.DataFrame) -> pd.DataFrame:
    x = rename_price(df)
    x = fix_price_room_count_from_pool(x)
    return add_unit_match_key(
        x,
        studio_typology_col="unit_typology",
        studio_typology_value="Студия",
        studio_key_room_count=1,
    )

In [6]:
price_pp = preprocess_price(price_raw)

## 4. Пересечение ключей price ↔ deals (до дедупликации)

Показывает, сколько уникальных `unit_match_key` совпадает между таблицами до любой фильтрации.

In [7]:
deals_pp = preprocess_deals(deals_raw)

price_keys = set(price_pp["unit_match_key"].unique())
deals_keys = set(deals_pp["unit_match_key"].unique())
common     = price_keys & deals_keys
union      = price_keys | deals_keys

print(f"Уникальных ключей в price:  {len(price_keys):>10,}")
print(f"Уникальных ключей в deals:  {len(deals_keys):>10,}")
print(f"Пересечение:                {len(common):>10,}  ({100*len(common)/len(union):.1f}% от объединения)")

Уникальных ключей в price:     882,647
Уникальных ключей в deals:     752,977
Пересечение:                   183,578  (12.6% от объединения)


## 5. Дедупликация сделок

Два уровня:
1. **`deal_row_key`** — удаляем строки с одинаковым ID сделки (точные дубли в источнике).
2. **`unit_match_key`** — оставляем одну строку на квартиру: ту, где `registration_date_raw` наименьшая (первая продажа).

In [8]:
# Примеры ключей с несколькими сделками
dup_keys = (
    deals_pp.groupby("unit_match_key")
    .size()
    .pipe(lambda s: s[s > 1])
    .sort_values(ascending=False)
    .head(2)
    .index
    .tolist()
)

show_cols = [c for c in ("unit_match_key", "deal_row_key", "registration_date_raw",
                          "floor", "area", "room_count", "price") if c in deals_pp.columns]
for key in dup_keys:
    print(f"\n{key}")
    print(deals_pp.loc[deals_pp["unit_match_key"] == key, show_cols].to_string(index=False))


1954@7465@@191@15@65,8@3
          unit_match_key  deal_row_key registration_date_raw floor area room_count  price
1954@7465@@191@15@65,8@3         87124            27.11.2023    15 65.8          3 433432
1954@7465@@191@15@65,8@3         87279            07.08.2023    15 65.8          3 446490
1954@7465@@191@15@65,8@3         87536            15.04.2022    15 65.8          3 485432
1954@7465@@191@15@65,8@3         87650            29.12.2021    15 65.8          3 391674

1954@7464@@154@14@65,2@
         unit_match_key  deal_row_key registration_date_raw floor area room_count  price
1954@7464@@154@14@65,2@         86946            20.12.2023    14 65.2          - 473792
1954@7464@@154@14@65,2@         86961            04.12.2023    14 65.2          - 431286
1954@7464@@154@14@65,2@         87551            06.04.2022    14 65.2          - 434090
1954@7464@@154@14@65,2@         87676            28.12.2021    14 65.2          - 382558


In [9]:
deals_pp["registration_date_raw"] = pd.to_datetime(
    deals_pp["registration_date_raw"], dayfirst=True, errors="coerce"
)

# Диагностика: пулы с >1 строкой
pool_sizes = deals_pp.groupby("unit_match_key").size()
n_multi = (pool_sizes > 1).sum()
print(f"deals строк: {len(deals_pp):,}  |  уникальных пулов: {pool_sizes.shape[0]:,}")
print(f"Пулов с >1 строкой: {n_multi:,} ({100*n_multi/pool_sizes.shape[0]:.1f}%)")
print("Распределение строк/пул:", pool_sizes.value_counts().sort_index().head(5).to_dict())

# Шаг 1: дедуп по deal_row_key
if "deal_row_key" in deals_pp.columns:
    n0 = len(deals_pp)
    deals_pp = deals_pp.drop_duplicates("deal_row_key", keep="first")
    print(f"\nДедуп deal_row_key: {n0:,} → {len(deals_pp):,} (−{n0-len(deals_pp):,})")

# Шаг 2: дедуп по unit_match_key — оставляем первую (наиболее раннюю) продажу
n1 = len(deals_pp)
deals_dedup = (
    deals_pp
    .sort_values("registration_date_raw", na_position="last")
    .drop_duplicates("unit_match_key", keep="first")
    .reset_index(drop=True)
)
print(f"Дедуп unit_match_key:  {n1:,} → {len(deals_dedup):,} (−{n1-len(deals_dedup):,})")

deals строк: 756,791  |  уникальных пулов: 752,977
Пулов с >1 строкой: 3,710 (0.5%)
Распределение строк/пул: {1: 749267, 2: 3617, 3: 82, 4: 11}

Дедуп deal_row_key: 756,791 → 756,791 (−0)
Дедуп unit_match_key:  756,791 → 752,977 (−3,814)


In [10]:
# Сравнение покрытия section: оригинал vs regex из premises_description
total = len(deals_dedup)

orig_notnull = deals_dedup["section"].notna().sum()
orig_filled  = deals_dedup["section"].replace("-", pd.NA).notna().sum()
print(f"section (оригинал):         {orig_notnull:>8,} / {total:,}  ({100*orig_notnull/total:.1f}%)  not-null")
print(f"section (без '-'):          {orig_filled:>8,} / {total:,}  ({100*orig_filled/total:.1f}%)  значимые")
print(deals_dedup["section"].value_counts().head(10).to_string())

parsed_series = deals_dedup["premises_description"].str.extract(r"секция\s+(\d+)", expand=False)
parsed = parsed_series.notna().sum()
print(f"\nsection_parsed (regex \\d+): {parsed:>8,} / {total:,}  ({100*parsed/total:.1f}%)")
print(parsed_series.value_counts().head(10).to_string())
print(f"\nПрирост vs not-null: +{parsed - orig_notnull:,}")
print(f"Прирост vs без '-':  +{parsed - orig_filled:,}")

section (оригинал):          421,632 / 752,977  (56.0%)  not-null
section (без '-'):           421,632 / 752,977  (56.0%)  значимые
section
1     143258
2      74733
3      55998
4      38587
5      28104
6      19259
7      15332
8      10023
9       7924
10      5538

section_parsed (regex \d+):  386,663 / 752,977  (51.4%)
premises_description
1     131504
2      68268
3      51606
4      35596
5      26096
6      17937
7      14351
8       9399
9       7500
10      5211

Прирост vs not-null: +-34,969
Прирост vs без '-':  +-34,969


## 6. JOIN + сохранение

**LEFT JOIN** — базовый вариант для модели вероятности продажи:  
каждая строка = один лот в один месяц; непроданные получают `registration_date_raw = NaT` → таргет = 0.

**INNER JOIN** — только реализованные лоты:  
используется для регрессионного анализа эластичности по ценам сделок.

In [11]:
from src.config import JOINED_LEFT_PARQUET, JOINED_INNER_PARQUET

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# 1. LEFT JOIN
joined_left = (
    price_pp.merge(
        deals_dedup,
        on="unit_match_key",
        how="left",
        suffixes=("_price", "_deals"),
    )
    .sort_values(["unit_match_key", "file_date"])
    .reset_index(drop=True)
)

# 2. INNER JOIN
joined_inner = (
    price_pp.merge(
        deals_dedup,
        on="unit_match_key",
        how="inner",
        suffixes=("_price", "_deals"),
    )
    .sort_values(["unit_match_key", "file_date"])
    .reset_index(drop=True)
)

print(f"joined_left:  {joined_left.shape}")
print(f"joined_inner: {joined_inner.shape}")

joined_left:  (4659378, 94)
joined_inner: (914441, 94)


In [12]:
# Приведение типов после JOIN
# float
_FLOAT_COLS = [
    "price_price", "area_price", "floor_price", "discount_pct", "exposure",
    "price_deals", "lat_price", "lng_price", "lat_deals", "lng_deals",
]
# datetime dayfirst — file_date и registration_date_raw в ЕДИНОМ формате
# (таргет sold_next_30d = разность этих дат, оба обязаны быть datetime64/dayfirst)
_DT_DAYFIRST = ["file_date", "sales_start_price", "registration_date_raw"]
# string (mixed int/str after merge — room_count comes as object with ints from deals)
_STR_COLS = ["room_count_price", "room_count_deals"]

def _cast_joined(df):
    out = df.copy()
    for col in _FLOAT_COLS:
        if col not in out.columns: continue
        out[col] = pd.to_numeric(
            out[col].astype(str).str.replace(",", ".", regex=False)
            if out[col].dtype == object else out[col],
            errors="coerce",
        )
    for col in _DT_DAYFIRST:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], dayfirst=True, errors="coerce")
    for col in _STR_COLS:
        if col in out.columns:
            out[col] = out[col].where(out[col].isna(), out[col].astype(str))
    return out

joined_left  = _cast_joined(joined_left)
joined_inner = _cast_joined(joined_inner)

print("Типы после каста:")
for col in _FLOAT_COLS + _DT_DAYFIRST + _STR_COLS:
    for df, name in ((joined_left, "left"), (joined_inner, "inner")):
        if col in df.columns:
            print(f"  {name:6s} {col:30s} {df[col].dtype}")
            break

Типы после каста:
  left   price_price                    Int64
  left   area_price                     Float64
  left   floor_price                    Int64
  left   discount_pct                   Int64
  left   exposure                       Int64
  left   price_deals                    float64
  left   lat_price                      Float64
  left   lng_price                      Float64
  left   lat_deals                      float64
  left   lng_deals                      float64
  left   file_date                      datetime64[us]
  left   sales_start_price              datetime64[us]
  left   registration_date_raw          datetime64[us]
  left   room_count_price               string
  left   room_count_deals               object


In [13]:
n_rows_total   = len(joined_left)
n_rows_sold    = joined_left["registration_date_raw"].notna().sum()
n_rows_unsold  = n_rows_total - n_rows_sold
pools_matched   = joined_left.loc[joined_left["registration_date_raw"].notna(), "unit_match_key"].nunique()
pools_unmatched = joined_left.loc[joined_left["registration_date_raw"].isna(),  "unit_match_key"].nunique()

joined_left.to_parquet(JOINED_LEFT_PARQUET, index=False)
joined_inner.to_parquet(JOINED_INNER_PARQUET, index=False)

print(f"LEFT JOIN ──────────────────────────────────────────────")
print(f"Сохранено: {JOINED_LEFT_PARQUET.name}")
print(f"  Всего строк:                      {n_rows_total:>10,}")
print(f"  Строк со сделкой  (sold=1):        {n_rows_sold:>9,}  ({100*n_rows_sold/n_rows_total:.1f}%)")
print(f"  Строк без сделки  (sold=0):        {n_rows_unsold:>9,}  ({100*n_rows_unsold/n_rows_total:.1f}%)")
print(f"  Уникальных пулов со сделкой:       {pools_matched:>9,}")
print(f"  Уникальных пулов без сделки:       {pools_unmatched:>9,}")
print(f"  Колонок:                           {joined_left.shape[1]:>9,}")

n_inner_pools = joined_inner["unit_match_key"].nunique()
print(f"INNER JOIN ─────────────────────────────────────────────")
print(f"Сохранено: {JOINED_INNER_PARQUET.name}")
print(f"  Всего строк:                      {len(joined_inner):>10,}")
print(f"  Уникальных пулов:                 {n_inner_pools:>10,}")
print(f"  Колонок:                          {joined_inner.shape[1]:>10,}")

print(f"Сводка ─────────────────────────────────────────────────")
print(f"  Пулов в прайсе:                   {price_pp['unit_match_key'].nunique():>10,}")
print(f"  Пулов в сделках (после дедуп):    {deals_dedup['unit_match_key'].nunique():>10,}")
print(f"  Матч (inner):                     {n_inner_pools:>10,}")
print(f"  Только в прайсе (left - inner):   {pools_unmatched:>10,}")


LEFT JOIN ──────────────────────────────────────────────
Сохранено: price_deals_left_unit_match_key.parquet
  Всего строк:                       4,659,378
  Строк со сделкой  (sold=1):          914,441  (19.6%)
  Строк без сделки  (sold=0):        3,744,937  (80.4%)
  Уникальных пулов со сделкой:         183,578
  Уникальных пулов без сделки:         699,069
  Колонок:                                  94
INNER JOIN ─────────────────────────────────────────────
Сохранено: price_deals_inner_unit_match_key.parquet
  Всего строк:                         914,441
  Уникальных пулов:                    183,578
  Колонок:                                  94
Сводка ─────────────────────────────────────────────────
  Пулов в прайсе:                      882,647
  Пулов в сделках (после дедуп):       752,977
  Матч (inner):                        183,578
  Только в прайсе (left - inner):      699,069


## 7. Контроль качества данных

Проверяем **left** и **inner** join на пропуски, диапазоны числовых колонок, аномальные значения категориальных полей и покрытие по срезам `file_date`.  
Таргет и фичи модели здесь не анализируются — только сырые данные после JOIN.

In [14]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Пропуски: сравнение left vs inner
def nan_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    total = len(df)
    r = pd.DataFrame({
        "n_nan":   df.isna().sum(),
        "pct_nan": df.isna().mean() * 100,
        "dtype":   df.dtypes.astype(str),
    })
    r.index.name = "column"
    r["name"] = name
    r["n_total"] = total
    return r[r["n_nan"] > 0].sort_values("pct_nan", ascending=False)

rep_left  = nan_report(joined_left,  "left")
rep_inner = nan_report(joined_inner, "inner")

# Только колонки с пропусками в любой из таблиц
all_cols = rep_left.index.union(rep_inner.index)
cmp = pd.DataFrame({
    "dtype":       joined_left.dtypes.reindex(all_cols).astype(str),
    "left  n_nan": rep_left["n_nan"].reindex(all_cols).fillna(0).astype(int),
    "left  %nan":  rep_left["pct_nan"].reindex(all_cols).fillna(0).round(1),
    "inner n_nan": rep_inner["n_nan"].reindex(all_cols).fillna(0).astype(int),
    "inner %nan":  rep_inner["pct_nan"].reindex(all_cols).fillna(0).round(1),
}).sort_values("left  %nan", ascending=False)

print(f"Колонок всего:  left={joined_left.shape[1]},  inner={joined_inner.shape[1]}")
print(f"Колонок с NaN:  left={len(rep_left)},  inner={len(rep_inner)}\n")
display(cmp.style
    .format({"left  %nan": "{:.1f}%", "inner %nan": "{:.1f}%"})
    .background_gradient(subset=["left  %nan", "inner %nan"], cmap="Reds", vmin=0, vmax=100)
)

Колонок всего:  left=94,  inner=94
Колонок с NaN:  left=66,  inner=14



,dtype,left n_nan,left %nan,inner n_nan,inner %nan
column,,,,,
section_deals,str,3849598,82.6%,104661,11.4%
area_deals,str,3744937,80.4%,0,0.0%
price_per_sqm,str,3744937,80.4%,0,0.0%
macro_district_deals,str,3744937,80.4%,0,0.0%
macro_district_direction,str,3744937,80.4%,0,0.0%
mortgage,float64,3744937,80.4%,0,0.0%
pledge_duration_months,str,3744937,80.4%,0,0.0%
pledge_holder_bank,str,3744937,80.4%,0,0.0%
pledge_type,str,3744937,80.4%,0,0.0%


In [15]:
# Числовые колонки: диапазоны и аномалии (left)
NUM_COLS = {
    "price_price":  "Цена (прайс)",
    "area_price":   "Площадь (прайс)",
    "floor_price":  "Этаж (прайс)",
    "discount_pct": "Скидка %",
    "exposure":     "Экспозиция, дн.",
}

def to_num_s(s: pd.Series) -> pd.Series:
    if s.dtype == object:
        return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")
    return pd.to_numeric(s, errors="coerce")

rows = []
for col, label in NUM_COLS.items():
    if col not in joined_left.columns:
        continue
    s = to_num_s(joined_left[col])
    rows.append({
        "колонка":   col,
        "label":     label,
        "n_valid":   int(s.notna().sum()),
        "n_zero":    int((s == 0).sum()),
        "n_neg":     int((s < 0).sum()),
        "min":       s.min(),
        "p1":        s.quantile(0.01),
        "median":    s.median(),
        "p99":       s.quantile(0.99),
        "max":       s.max(),
    })

num_stats = pd.DataFrame(rows).set_index("label")
display(num_stats.drop(columns="колонка").style
    .format({
        "n_valid": "{:,}", "n_zero": "{:,}", "n_neg": "{:,}",
        "min": "{:,.1f}", "p1": "{:,.1f}", "median": "{:,.1f}",
        "p99": "{:,.1f}", "max": "{:,.1f}",
    })
    .highlight_between(subset=["n_zero", "n_neg"], left=1, color="#ffd6d6")
    .set_caption("Числовые колонки (LEFT JOIN): диапазоны и аномалии")
)

# Флаги
for _, row in pd.DataFrame(rows).iterrows():
    flags = []
    if row["n_zero"] > 0:  flags.append(f"⚠ нулей: {int(row['n_zero']):,}")
    if row["n_neg"]  > 0:  flags.append(f"⚠ отрицательных: {int(row['n_neg']):,}")
    if flags:
        print(f"{row['колонка']:20s}  {' | '.join(flags)}")

,n_valid,n_zero,n_neg,min,p1,median,p99,max
label,,,,,,,,
Цена (прайс),"4,659,378",0,0,"38,572.0","95,778.0","254,058.0","683,190.0","2,727,180.0"
Площадь (прайс),"4,659,378",0,0,2.2,19.9,44.8,117.5,491.7
Этаж (прайс),"4,659,378",0,0,1.0,1.0,10.0,40.0,67.0
Скидка %,"4,659,377","3,660,977",27,-4.0,0.0,0.0,24.0,60.0
"Экспозиция, дн.","4,138,657","537,371",0,0.0,0.0,103.0,"1,125.0","8,271.0"


discount_pct          ⚠ нулей: 3,660,977 | ⚠ отрицательных: 27
exposure              ⚠ нулей: 537,371


In [16]:
# Категориальные колонки: вариации и неожиданные значения (left)
CAT_COLS = [
    ("room_count_price",    "Комнатность"),
    ("project_class_price", "Класс проекта"),
    ("region_price",        "Регион"),
    ("premises_type_price", "Тип помещения"),
    ("keys_status_price",   "Ключи"),
    ("finish_tier",         "Отделка К"),
    ("contract_type_k",     "Договор К"),
    ("stage_k",             "Стадия К"),
]

print("Категориальные колонки (LEFT JOIN): уникальные значения\n")
for col, label in CAT_COLS:
    if col not in joined_left.columns:
        print(f"  {label}: колонка отсутствует"); continue
    vc = joined_left[col].astype(str).value_counts(dropna=False)
    n_unique = joined_left[col].nunique(dropna=False)
    nan_pct  = joined_left[col].isna().mean() * 100
    print(f"{'─'*60}")
    print(f"  {label} ({col})   уникальных={n_unique}   NaN={nan_pct:.1f}%")
    print(vc.head(15).to_string())
    print()

Категориальные колонки (LEFT JOIN): уникальные значения

────────────────────────────────────────────────────────────
  Комнатность (room_count_price)   уникальных=6   NaN=0.0%
room_count_price
1    2180667
2    1614146
3     728348
4     135836
5        369
6         12

────────────────────────────────────────────────────────────
  Класс проекта (project_class_price)   уникальных=2   NaN=0.0%
project_class_price
комфорт    3724298
бизнес      935080

────────────────────────────────────────────────────────────
  Регион (region_price)   уникальных=3   NaN=0.0%
region_price
Москва                2019601
Московская область    1827778
Новая Москва           811999

────────────────────────────────────────────────────────────
  Тип помещения (premises_type_price)   уникальных=2   NaN=0.0%
premises_type_price
Квартира      4300577
Апартамент     358801

────────────────────────────────────────────────────────────
  Ключи (keys_status_price)   уникальных=82   NaN=0.0%
keys_status_price
2Q 2